<a href="https://colab.research.google.com/github/Peeyusj/FashionMNIST/blob/main/FashionMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [12]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

In [13]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size,shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [14]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [15]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [16]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [17]:
train(train_dataloader, model, loss_fn, optimizer)

loss: 2.302584  [   64/60000]
loss: 2.299910  [ 6464/60000]
loss: 2.272901  [12864/60000]
loss: 2.280246  [19264/60000]
loss: 2.236943  [25664/60000]
loss: 2.233731  [32064/60000]
loss: 2.220893  [38464/60000]
loss: 2.192113  [44864/60000]
loss: 2.173032  [51264/60000]
loss: 2.164426  [57664/60000]


In [18]:
def test(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.eval()
    correct = 0
    with torch.no_grad():
      for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        correct += (pred.argmax(1) == y).sum().item()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

    correct /= size
    print(f"Test Accuracy: {(100*correct):>0.1f}%")

In [19]:
for epoch in range(5):
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn, optimizer)

loss: 2.166609  [   64/60000]
loss: 2.128883  [ 6464/60000]
loss: 2.136085  [12864/60000]
loss: 2.035532  [19264/60000]
loss: 2.085313  [25664/60000]
loss: 2.020710  [32064/60000]
loss: 1.990969  [38464/60000]
loss: 2.009240  [44864/60000]
loss: 1.948406  [51264/60000]
loss: 1.832256  [57664/60000]
loss: 1.887313  [   64/10000]
loss: 1.887807  [ 6464/10000]
Test Accuracy: 51.5%
loss: 1.920479  [   64/60000]
loss: 1.845798  [ 6464/60000]
loss: 1.786930  [12864/60000]
loss: 1.712301  [19264/60000]
loss: 1.717680  [25664/60000]
loss: 1.672010  [32064/60000]
loss: 1.554740  [38464/60000]
loss: 1.506735  [44864/60000]
loss: 1.572631  [51264/60000]
loss: 1.474192  [57664/60000]
loss: 1.509574  [   64/10000]
loss: 1.533208  [ 6464/10000]
Test Accuracy: 59.9%
loss: 1.534303  [   64/60000]
loss: 1.492262  [ 6464/60000]
loss: 1.429712  [12864/60000]
loss: 1.443252  [19264/60000]
loss: 1.304210  [25664/60000]
loss: 1.296821  [32064/60000]
loss: 1.403820  [38464/60000]
loss: 1.352378  [44864/60000

In [20]:
torch.save(model.state_dict(), "fashion_classifier.pth")

In [21]:
from PIL import Image
img = Image.open("item.jpg")
print(img.size, img.mode)

FileNotFoundError: [Errno 2] No such file or directory: 'item.jpg'